# Summary_Day12.ipynb  
## 심화 경사하강법과 옵티마이저 구현하기

이번 12강은 **경사하강법과 대표 Optimizer를 NumPy로 직접 구현하는 심화 실습**이다.

앞 강의에서는 `optim.SGD`, `optim.Adam`, `optim.AdamW`처럼 PyTorch에 이미 만들어진 Optimizer를 사용했다.  
이번 강의에서는 그 안에서 실제로 어떤 계산이 일어나는지 직접 코드로 만든다.

전체 흐름은 다음이다.

```text
경사하강법 복습
→ 선형회귀 y = Wx + b를 NumPy로 학습
→ cost_function으로 MSE 계산
→ gradient 함수로 dW, db 계산
→ SGD 직접 구현
→ Momentum 직접 구현
→ RMSprop 직접 구현
→ Adam 직접 구현
→ AdamW 직접 구현
→ 같은 문제에서 Optimizer별 이동 경로 비교
```

> 필기 포인트:  
> Optimizer는 결국 `params`를 `grads` 방향에 따라 어떻게 움직일지 정하는 업데이트 규칙이다.  
> 즉, Optimizer의 차이는 “기울기를 보고 얼마나, 어떤 방식으로 움직일 것인가”의 차이다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. 경사하강법의 기본 업데이트 식을 직접 구현한다.
2. `W`, `b`가 MSE를 줄이는 방향으로 바뀌는 흐름을 확인한다.
3. `params`, `grads` 딕셔너리 구조로 Optimizer를 일반화한다.
4. SGD, Momentum, RMSprop, Adam, AdamW의 업데이트 차이를 이해한다.
5. 각 Optimizer가 같은 손실 곡면에서 어떻게 이동하는지 그래프로 비교한다.
6. 변수명과 약어 의미를 정리해 PyTorch Optimizer를 볼 때도 내부 흐름을 떠올릴 수 있게 한다.

> 이번 노트의 핵심 질문:  
> `optimizer.step()` 한 줄 안에서는 실제로 어떤 계산이 일어나는가?

## 2. 라이브러리 준비

이번 실습은 Optimizer 내부 계산을 직접 보기 위해 NumPy 중심으로 진행한다.  
그래프 확인을 위해 Matplotlib도 사용한다.

### 함수/모듈 사용법

```python
import numpy as np
import matplotlib.pyplot as plt
```

- `np`: NumPy 약어다. 배열 계산, 합계, 제곱근, 0 배열 생성 등에 사용한다.
- `plt`: Matplotlib의 pyplot 약어다. 손실 곡선과 Optimizer 이동 경로를 그릴 때 사용한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=6)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("NumPy version:", np.__version__)

## 3. 경사하강법 기본 개념 복습

경사하강법은 손실함수의 기울기를 이용해 손실이 작아지는 방향으로 파라미터를 이동시키는 방법이다.

기본 업데이트 식은 다음이다.

```text
새 파라미터 = 현재 파라미터 - learning_rate × gradient
```

선형회귀에서는 다음 두 파라미터를 학습한다.

```text
W: weight, 가중치
b: bias, 편향
```

이번 예제의 목표는 다음 관계를 학습하는 것이다.

```text
X = [1, 2, 3, 4, 5]
y = [2, 4, 6, 8, 10]
정답 관계: y = 2X
```

따라서 이상적인 값은 대략 다음이다.

```text
W ≈ 2
b ≈ 0
```

In [ ]:
X = np.array([1, 2, 3, 4, 5])
y = np.array([2, 4, 6, 8, 10])

W = 0.0
b = 0.0

learning_rate = 0.01
iterations = 1000

print("X:", X)
print("y:", y)
print("초기 W:", W)
print("초기 b:", b)
print("learning_rate:", learning_rate)
print("iterations:", iterations)

## 4. 비용 함수 MSE 만들기

이번 강의의 비용 함수는 MSE다.

```text
MSE = mean((prediction - y)²)
```

### 함수 사용법

```python
cost_function(X, y, W, b)
```

- `X`: 입력 데이터다.
- `y`: 정답 데이터다.
- `W`: 현재 가중치다.
- `b`: 현재 편향이다.
- 반환값은 현재 `W`, `b`에서의 평균 제곱 오차다.

> 여기서는 `prediction = W * X + b`로 예측값을 만든다.

In [ ]:
def cost_function(X, y, W, b):
    n = len(X)
    prediction = W * X + b
    cost = np.sum((prediction - y) ** 2) / n
    return cost

initial_cost = cost_function(X, y, W, b)

print("초기 cost:", initial_cost)

코드 한 줄씩 의미:

```python
n = len(X)
```

데이터 개수를 구한다.

```python
prediction = W * X + b
```

현재 파라미터로 예측값을 계산한다.

```python
cost = np.sum((prediction - y) ** 2) / n
```

예측값과 정답의 차이를 제곱하고 평균낸다.

```python
return cost
```

계산된 MSE 값을 반환한다.

## 5. 기울기 함수 만들기

경사하강법은 손실함수의 기울기를 알아야 한다.

선형회귀 MSE에서 필요한 기울기는 두 개다.

```text
dW: W에 대한 손실의 기울기
db: b에 대한 손실의 기울기
```

### 함수 사용법

```python
gradient(X, y, W, b)
```

- 현재 `W`, `b`에서의 `dW`, `db`를 반환한다.
- `dW`, `db`는 파라미터를 어느 방향으로 바꿔야 하는지 알려준다.

In [ ]:
def gradient(X, y, W, b):
    n = len(X)
    prediction = W * X + b

    dW = np.sum(2 * X * (prediction - y)) / n
    db = np.sum(2 * (prediction - y)) / n

    return dW, db

dW, db = gradient(X, y, W, b)

print("초기 dW:", dW)
print("초기 db:", db)

코드 한 줄씩 의미:

```python
prediction = W * X + b
```

현재 모델의 예측값을 계산한다.

```python
dW = np.sum(2 * X * (prediction - y)) / n
```

MSE를 `W`에 대해 미분한 값을 계산한다.

```python
db = np.sum(2 * (prediction - y)) / n
```

MSE를 `b`에 대해 미분한 값을 계산한다.

```python
return dW, db
```

두 기울기를 함께 반환한다.

> 기억할 점:  
> `dW`, `db`가 크면 현재 파라미터가 많이 수정될 가능성이 크다.  
> learning rate가 너무 크면 이 수정 폭이 커져 발산할 수 있다.

## 6. 경사하강법 학습 루프 구현

이제 `iterations`만큼 반복하면서 `W`, `b`를 업데이트한다.

기본 흐름은 다음이다.

```text
1. 현재 W, b에서 gradient 계산
2. W = W - learning_rate × dW
3. b = b - learning_rate × db
4. cost 기록
```

In [ ]:
W = 0.0
b = 0.0

history = []

for i in range(iterations):
    dW, db = gradient(X, y, W, b)

    W = W - learning_rate * dW
    b = b - learning_rate * db

    cost = cost_function(X, y, W, b)
    history.append([i, cost, W, b])

    if i % 100 == 0:
        print(f"Iteration {i}: Cost = {cost:.6f}, W = {W:.6f}, b = {b:.6f}")

history = np.array(history)

print(f"Final parameters: W = {W:.6f}, b = {b:.6f}")

코드 한 줄씩 의미:

```python
history = []
```

반복마다 cost, W, b를 저장할 리스트다.

```python
for i in range(iterations):
```

정해진 반복 횟수만큼 학습한다.

```python
dW, db = gradient(X, y, W, b)
```

현재 위치에서 기울기를 계산한다.

```python
W = W - learning_rate * dW
b = b - learning_rate * db
```

기울기의 반대 방향으로 파라미터를 이동한다.

```python
history.append([i, cost, W, b])
```

나중에 그래프를 그리기 위해 값을 저장한다.

## 7. 손실 감소 그래프와 최종 직선 확인

출력 숫자만 보면 학습이 잘 되는지 감이 약할 수 있다.  
그래프로 보면 cost가 줄어드는 흐름과 최종 예측 직선을 더 쉽게 볼 수 있다.

In [ ]:
plt.plot(history[:, 0], history[:, 1])
plt.xlabel("iteration")
plt.ylabel("cost")
plt.title("Gradient Descent Cost")
plt.show()

In [ ]:
y_pred = W * X + b

plt.scatter(X, y, label="true data")
plt.plot(X, y_pred, label="learned line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Learned Linear Function")
plt.legend()
plt.show()

print("y_true:", y)
print("y_pred:", np.round(y_pred, 4))

그래프 해석:

- cost 그래프가 내려가면 학습이 진행되는 것이다.
- 최종 직선이 점들을 잘 지나가면 `W`, `b`가 적절히 학습된 것이다.
- 이번 데이터의 정답 관계는 `y = 2X`라서 `W`는 2에 가까워지고 `b`는 0에 가까워진다.

## 8. Optimizer 공통 구조: params와 grads

강의의 Optimizer 구현은 `params`와 `grads` 딕셔너리를 사용한다.

```text
params: 현재 파라미터 값
grads: 각 파라미터에 대한 기울기
```

예시는 다음과 같다.

```python
params = {"W": ..., "b": ...}
grads = {"W": ..., "b": ...}
```

이 구조를 쓰면 W, b뿐 아니라 여러 Layer의 weight, bias도 같은 방식으로 업데이트할 수 있다.

> PyTorch의 `model.parameters()`도 결국 모델 안의 여러 파라미터를 Optimizer에게 넘기는 구조라고 보면 된다.

In [ ]:
params = {
    "W": np.array(0.0),
    "b": np.array(0.0)
}

grads = {
    "W": np.array(-20.0),
    "b": np.array(-6.0)
}

print("params:", params)
print("grads:", grads)

## 9. SGD 구현

SGD는 Stochastic Gradient Descent의 약자다.

가장 기본적인 업데이트 식은 다음이다.

```text
param = param - learning_rate × gradient
```

### 클래스 사용법

```python
optimizer = SGD(learning_rate=0.01)
optimizer.update(params, grads)
```

- `learning_rate`: 한 번에 얼마나 움직일지 정하는 값이다.
- `params`: 수정할 파라미터 딕셔너리다.
- `grads`: 각 파라미터의 기울기 딕셔너리다.

In [ ]:
class SGD:
    def __init__(self, learning_rate=0.01):
        self.lr = learning_rate

    def update(self, params, grads):
        for key in params.keys():
            params[key] -= self.lr * grads[key]

코드 한 줄씩 의미:

```python
class SGD:
```

SGD Optimizer 클래스를 만든다.

```python
def __init__(self, learning_rate=0.01):
```

객체를 만들 때 learning rate를 받는다.

```python
self.lr = learning_rate
```

learning rate를 객체 내부에 저장한다.

```python
for key in params.keys():
```

`W`, `b` 같은 모든 파라미터 이름을 반복한다.

```python
params[key] -= self.lr * grads[key]
```

각 파라미터를 기울기의 반대 방향으로 업데이트한다.

In [ ]:
params_sgd = {
    "W": np.array(0.0),
    "b": np.array(0.0)
}

grads_sgd = {
    "W": np.array(-20.0),
    "b": np.array(-6.0)
}

optimizer_sgd = SGD(learning_rate=0.01)
optimizer_sgd.update(params_sgd, grads_sgd)

print("SGD update 후 params:")
print(params_sgd)

## 10. Momentum 구현

Momentum은 이전 업데이트 방향을 기억해서 관성을 붙이는 방식이다.

비유하면 공이 그릇을 따라 굴러가며 점점 속도가 붙는 느낌이다.

기본 구조는 다음이다.

```text
v = momentum × v - learning_rate × gradient
param = param + v
```

여기서 `v`는 velocity, 즉 속도다.

### 클래스 사용법

```python
optimizer = Momentum(learning_rate=0.01, momentum=0.9)
optimizer.update(params, grads)
```

In [ ]:
class Momentum:
    def __init__(self, learning_rate=0.01, momentum=0.9):
        self.lr = learning_rate
        self.momentum = momentum
        self.v = None

    def update(self, params, grads):
        if self.v is None:
            self.v = {}

            for key, val in params.items():
                self.v[key] = np.zeros_like(val)

        for key in params.keys():
            self.v[key] = self.momentum * self.v[key] - self.lr * grads[key]
            params[key] += self.v[key]

코드 한 줄씩 의미:

```python
self.v = None
```

처음에는 velocity 저장소가 없다는 뜻이다.

```python
if self.v is None:
```

첫 업데이트 때 velocity 딕셔너리를 만든다.

```python
self.v[key] = np.zeros_like(val)
```

각 파라미터와 같은 shape의 0 배열을 만든다.

```python
self.v[key] = self.momentum * self.v[key] - self.lr * grads[key]
```

이전 속도에 관성을 곱하고 현재 gradient 방향을 반영한다.

```python
params[key] += self.v[key]
```

계산된 velocity만큼 파라미터를 이동한다.

> 핵심:  
> Momentum은 현재 gradient만 보지 않고 이전 이동 방향도 함께 본다.

In [ ]:
params_momentum = {
    "W": np.array(0.0),
    "b": np.array(0.0)
}

grads_momentum = {
    "W": np.array(-20.0),
    "b": np.array(-6.0)
}

optimizer_momentum = Momentum(learning_rate=0.01, momentum=0.9)

for step in range(3):
    optimizer_momentum.update(params_momentum, grads_momentum)
    print(f"step {step + 1}:", params_momentum)

## 11. Momentum 주요 변수 정리

| 변수 | 뜻 |
|---|---|
| `W` | weight, 가중치다 |
| `b` | bias, 편향이다 |
| `v` | velocity, 이전 이동 방향과 속도다 |
| `momentum` | 이전 속도를 얼마나 유지할지 정하는 값이다 |
| `theta` | 모든 학습 대상 파라미터를 통틀어 부르는 기호다 |

Momentum은 `W`, `b`를 직접 업데이트하는 것이 아니라, 먼저 이동량 `v`를 계산한 뒤 그 이동량을 파라미터에 더한다.

## 12. RMSprop 구현

RMSprop은 파라미터마다 학습률을 다르게 조절하는 Optimizer다.

핵심 아이디어는 다음이다.

```text
최근 gradient 제곱의 이동 평균을 저장한다.
gradient가 자주 크게 나오는 방향은 작게 움직인다.
gradient가 작게 나오는 방향은 상대적으로 크게 움직인다.
```

기본 구조는 다음이다.

```text
h = decay_rate × h + (1 - decay_rate) × gradient²
param = param - learning_rate × gradient / (sqrt(h) + epsilon)
```

### 클래스 사용법

```python
optimizer = RMSprop(learning_rate=0.01, decay_rate=0.99)
optimizer.update(params, grads)
```

In [ ]:
class RMSprop:
    def __init__(self, learning_rate=0.01, decay_rate=0.99):
        self.lr = learning_rate
        self.decay_rate = decay_rate
        self.h = None

    def update(self, params, grads):
        if self.h is None:
            self.h = {}

            for key, val in params.items():
                self.h[key] = np.zeros_like(val)

        for key in params.keys():
            self.h[key] *= self.decay_rate
            self.h[key] += (1 - self.decay_rate) * (grads[key] ** 2)

            params[key] -= self.lr * grads[key] / (np.sqrt(self.h[key]) + 1e-7)

코드 한 줄씩 의미:

```python
self.h = None
```

gradient 제곱의 이동 평균 저장소를 아직 만들지 않았다는 뜻이다.

```python
self.h[key] = np.zeros_like(val)
```

파라미터와 같은 shape의 0 배열을 만든다.

```python
self.h[key] *= self.decay_rate
```

이전 gradient 제곱 평균의 영향력을 조금 줄인다.

```python
self.h[key] += (1 - self.decay_rate) * (grads[key] ** 2)
```

현재 gradient 제곱을 일정 비율로 추가한다.

```python
params[key] -= self.lr * grads[key] / (np.sqrt(self.h[key]) + 1e-7)
```

gradient를 `sqrt(h)`로 나누어 파라미터별 이동 크기를 조절한다.

> `1e-7`은 0으로 나누는 문제를 막기 위한 아주 작은 값이다.

In [ ]:
params_rms = {
    "W": np.array(0.0),
    "b": np.array(0.0)
}

grads_rms = {
    "W": np.array(-20.0),
    "b": np.array(-6.0)
}

optimizer_rms = RMSprop(learning_rate=0.01, decay_rate=0.99)

for step in range(3):
    optimizer_rms.update(params_rms, grads_rms)
    print(f"step {step + 1}:", params_rms)

## 13. RMSprop 주요 변수 정리

| 변수 | 뜻 |
|---|---|
| `h` | gradient 제곱의 지수 이동 평균이다 |
| `decay_rate` | 이전 값을 얼마나 유지할지 정하는 감쇠율이다 |
| `gt` | 현재 gradient를 뜻한다 |
| `epsilon` | 0으로 나누는 것을 막는 작은 값이다 |
| `np.sqrt()` | 제곱근을 계산한다 |

지수 이동 평균은 최근 값에 더 큰 비중을 두고 오래된 값은 점점 잊는 평균 방식이다.

예를 들어 decay가 0.9라면 다음 감각이다.

```text
현재 값 반영: 10%
이전 값 유지: 90%
더 오래된 값은 0.9가 계속 곱해져 영향이 작아짐
```

## 14. Adam 구현

Adam은 Momentum과 RMSprop을 결합한 Optimizer다.

Adam은 두 가지 정보를 함께 사용한다.

```text
m: gradient의 지수 이동 평균, 방향 정보
v: gradient 제곱의 지수 이동 평균, 크기 정보
```

그리고 학습 초기에 `m`, `v`가 0에서 시작해 작게 치우치는 문제를 보정한다.

```text
m_hat = m / (1 - beta1^t)
v_hat = v / (1 - beta2^t)
```

### 클래스 사용법

```python
optimizer = Adam(learning_rate=0.001, beta1=0.9, beta2=0.999)
optimizer.update(params, grads)
```

In [ ]:
class Adam:
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.iter = 0
        self.m = None
        self.v = None

    def update(self, params, grads):
        if self.m is None or self.v is None:
            self.m = {}
            self.v = {}

            for key, val in params.items():
                self.m[key] = np.zeros_like(val)
                self.v[key] = np.zeros_like(val)

        self.iter += 1

        for key in params.keys():
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)

            m_hat = self.m[key] / (1 - self.beta1 ** self.iter)
            v_hat = self.v[key] / (1 - self.beta2 ** self.iter)

            params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + 1e-7)

코드 한 줄씩 의미:

```python
self.beta1
```

1차 모멘트 `m`의 이동 평균 비율이다.

```python
self.beta2
```

2차 모멘트 `v`의 이동 평균 비율이다.

```python
self.iter += 1
```

현재 update step 번호를 1 증가시킨다.

```python
self.m[key] = beta1 * m + (1 - beta1) * grad
```

최근 gradient 방향을 누적한다.

```python
self.v[key] = beta2 * v + (1 - beta2) * grad²
```

최근 gradient 크기를 누적한다.

```python
m_hat = m / (1 - beta1 ** iter)
v_hat = v / (1 - beta2 ** iter)
```

학습 초기에 0으로 치우치는 값을 보정한다.

```python
params[key] -= lr * m_hat / (sqrt(v_hat) + epsilon)
```

보정된 방향과 크기 정보를 이용해 파라미터를 업데이트한다.

In [ ]:
params_adam = {
    "W": np.array(0.0),
    "b": np.array(0.0)
}

grads_adam = {
    "W": np.array(-20.0),
    "b": np.array(-6.0)
}

optimizer_adam = Adam(learning_rate=0.01)

for step in range(3):
    optimizer_adam.update(params_adam, grads_adam)
    print(f"step {step + 1}:", params_adam)

## 15. Adam 주요 변수 정리

| 변수 | 뜻 |
|---|---|
| `m` | 1차 모멘트, gradient의 지수 이동 평균이다 |
| `v` | 2차 모멘트, gradient 제곱의 지수 이동 평균이다 |
| `m_hat` | 편향 보정된 1차 모멘트다 |
| `v_hat` | 편향 보정된 2차 모멘트다 |
| `beta1` | `m`의 이동 평균 비율이다 |
| `beta2` | `v`의 이동 평균 비율이다 |
| `iter` | update step 수다 |

정리하면 Adam은 다음 두 질문을 동시에 본다.

```text
최근에 어느 방향으로 움직였는가? → m, Momentum 계열
얼마나 크게 움직였는가? → v, RMSprop 계열
```

## 16. AdamW 구현

AdamW는 Adam에 weight decay를 분리해서 적용한 방식이다.

강의 원본 코드의 마지막 줄에는 `lr_t`라는 변수가 등장한다.  
하지만 해당 코드 안에서 `lr_t`가 정의되어 있지 않아서 그대로 실행하면 에러가 날 수 있다.

이 Summary에서는 실행 가능하도록 `self.lr`를 사용해 AdamW를 구현한다.

AdamW의 핵심은 다음이다.

```text
1. Adam 방식으로 m, v를 계산한다.
2. weight_decay로 파라미터 자체를 조금 줄인다.
3. Adam update를 적용한다.
```

### 클래스 사용법

```python
optimizer = AdamW(learning_rate=0.001, weight_decay=0.01)
optimizer.update(params, grads)
```

In [ ]:
class AdamW:
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, weight_decay=0.01):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.iter = 0
        self.m = None
        self.v = None

    def update(self, params, grads):
        if self.m is None or self.v is None:
            self.m = {}
            self.v = {}

            for key, val in params.items():
                self.m[key] = np.zeros_like(val)
                self.v[key] = np.zeros_like(val)

        self.iter += 1

        for key in params.keys():
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)

            m_hat = self.m[key] / (1 - self.beta1 ** self.iter)
            v_hat = self.v[key] / (1 - self.beta2 ** self.iter)

            params[key] -= self.lr * self.weight_decay * params[key]
            params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + 1e-7)

코드 한 줄씩 의미:

```python
self.weight_decay = weight_decay
```

가중치 감쇠 정도를 저장한다.

```python
params[key] -= self.lr * self.weight_decay * params[key]
```

파라미터 자체를 조금 줄인다.  
가중치가 너무 커지는 것을 막아 과적합을 줄이는 데 도움을 준다.

```python
params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + 1e-7)
```

Adam 방식의 업데이트를 적용한다.

> 필기 포인트:  
> AdamW는 Adam의 gradient update와 weight decay를 분리해 적용하는 방식이라고 기억하면 된다.

In [ ]:
params_adamw = {
    "W": np.array(1.0),
    "b": np.array(1.0)
}

grads_adamw = {
    "W": np.array(-20.0),
    "b": np.array(-6.0)
}

optimizer_adamw = AdamW(learning_rate=0.01, weight_decay=0.01)

for step in range(3):
    optimizer_adamw.update(params_adamw, grads_adamw)
    print(f"step {step + 1}:", params_adamw)

## 17. Optimizer별 이동 경로 비교용 함수 만들기

같은 손실 곡면에서 Optimizer들이 어떻게 움직이는지 비교한다.

여기서는 간단한 2차 함수 손실을 사용한다.

```text
loss(W, b) = (W - 3)² + 0.5 × (b + 2)²
```

최솟값은 다음 위치다.

```text
W = 3
b = -2
```

각 Optimizer가 이 위치를 향해 어떻게 이동하는지 본다.

In [ ]:
def quadratic_loss_and_grads(params):
    W = params["W"]
    b = params["b"]

    loss = (W - 3.0) ** 2 + 0.5 * (b + 2.0) ** 2

    grads = {
        "W": 2.0 * (W - 3.0),
        "b": (b + 2.0)
    }

    return float(loss), grads


def run_optimizer(optimizer, steps=60):
    params = {
        "W": np.array(-4.0),
        "b": np.array(4.0)
    }

    history = []

    for step in range(steps):
        loss, grads = quadratic_loss_and_grads(params)

        history.append([
            step,
            loss,
            params["W"].item(),
            params["b"].item()
        ])

        optimizer.update(params, grads)

    return np.array(history)

함수 설명:

```python
quadratic_loss_and_grads(params)
```

현재 `W`, `b`에서 loss와 gradient를 계산한다.

```python
run_optimizer(optimizer, steps=60)
```

주어진 Optimizer를 같은 시작점에서 여러 step 실행하고 이동 기록을 반환한다.

> 이 비교는 실제 딥러닝 전체 학습을 대체하는 것이 아니라, Optimizer 업데이트 규칙 차이를 눈으로 보기 위한 작은 실험이다.

## 18. Optimizer별 손실 곡선 비교

SGD, Momentum, RMSprop, Adam, AdamW를 같은 문제에서 비교한다.

In [ ]:
optimizers = {
    "SGD": SGD(learning_rate=0.08),
    "Momentum": Momentum(learning_rate=0.08, momentum=0.9),
    "RMSprop": RMSprop(learning_rate=0.05, decay_rate=0.99),
    "Adam": Adam(learning_rate=0.1),
    "AdamW": AdamW(learning_rate=0.1, weight_decay=0.01)
}

optimizer_histories = {}

for name, optimizer in optimizers.items():
    optimizer_histories[name] = run_optimizer(optimizer, steps=60)

for name, hist in optimizer_histories.items():
    plt.plot(hist[:, 0], hist[:, 1], label=name)

plt.xlabel("step")
plt.ylabel("loss")
plt.title("Optimizer Loss Comparison")
plt.legend()
plt.show()

그래프 해석:

- loss가 빠르게 내려갈수록 해당 예제에서는 빠르게 최솟값에 접근한 것이다.
- Momentum은 이전 이동 방향을 기억해 더 빠르게 움직일 수 있다.
- RMSprop은 파라미터별 gradient 크기를 보정한다.
- Adam은 방향 정보와 크기 정보를 함께 사용한다.
- AdamW는 Adam에 weight decay를 분리 적용한다.

## 19. Optimizer별 이동 경로 비교

이번에는 손실 곡면 위에서 `W`, `b`가 어떻게 움직이는지 경로로 본다.

In [ ]:
W_grid = np.linspace(-5, 4, 120)
b_grid = np.linspace(-4, 5, 120)

WW, BB = np.meshgrid(W_grid, b_grid)
ZZ = (WW - 3.0) ** 2 + 0.5 * (BB + 2.0) ** 2

plt.contour(WW, BB, ZZ, levels=30)

for name, hist in optimizer_histories.items():
    plt.plot(hist[:, 2], hist[:, 3], marker="o", markersize=2, label=name)

plt.scatter([3.0], [-2.0], marker="*", s=150, label="minimum")

plt.xlabel("W")
plt.ylabel("b")
plt.title("Optimizer Paths on Loss Surface")
plt.legend()
plt.show()

그래프 해석:

- 별표는 최솟값 위치다.
- 각 선은 Optimizer가 `W`, `b`를 업데이트하며 이동한 경로다.
- 이동 경로가 매끄럽거나 빠르게 최솟값 근처로 가면 해당 예제에서는 안정적인 업데이트라고 볼 수 있다.
- 실제 딥러닝에서는 손실 곡면이 훨씬 복잡하므로 항상 같은 결과가 나오지는 않는다.

## 20. 각 Optimizer를 언제 떠올리면 좋은가

| Optimizer | 핵심 아이디어 | 기억할 포인트 |
|---|---|---|
| SGD | 현재 gradient만 보고 이동 | 가장 기본이다 |
| Momentum | 이전 이동 방향을 기억 | 관성, velocity |
| RMSprop | gradient 제곱 평균으로 학습률 조절 | 파라미터별 보폭 조절 |
| Adam | Momentum + RMSprop | 방향과 크기 모두 사용 |
| AdamW | Adam + 분리된 weight decay | 일반화와 과적합 완화 |

> 실무 감각:  
> 딥러닝에서는 AdamW가 자주 쓰이고, 기본 개념을 이해할 때는 SGD와 Momentum을 먼저 잡는 것이 좋다.

In [ ]:
optimizer_summary = {
    "SGD": "현재 gradient만 사용한다",
    "Momentum": "이전 이동 방향을 velocity로 기억한다",
    "RMSprop": "gradient 제곱 이동 평균으로 파라미터별 보폭을 조절한다",
    "Adam": "Momentum과 RMSprop을 결합한다",
    "AdamW": "Adam에 weight decay를 분리 적용한다"
}

for name, desc in optimizer_summary.items():
    print(f"{name}: {desc}")

## 21. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `np` | NumPy 약어 | 배열 계산에 사용한다 |
| `plt` | Matplotlib pyplot 약어 | 그래프를 그린다 |
| `X` | 입력 데이터 | 모델에 들어가는 값이다 |
| `y` | 정답 데이터 | 모델이 맞혀야 하는 값이다 |
| `W` | weight, 가중치 | 입력에 곱해지는 학습 파라미터다 |
| `b` | bias, 편향 | 예측값에 더해지는 학습 파라미터다 |
| `prediction` | 예측값 | `W * X + b` |
| `cost` | 비용, 손실 | 여기서는 MSE다 |
| `MSE` | Mean Squared Error | 평균 제곱 오차다 |
| `gradient` | 기울기 | 손실이 커지는 방향이다 |
| `dW` | W에 대한 기울기 | W 업데이트에 사용한다 |
| `db` | b에 대한 기울기 | b 업데이트에 사용한다 |
| `lr` | learning rate | 한 step 이동 크기다 |
| `iterations` | 반복 횟수 | 경사하강법 반복 수다 |
| `params` | 파라미터 딕셔너리 | W, b 등을 담는다 |
| `grads` | 기울기 딕셔너리 | dW, db 등을 담는다 |
| `key` | 딕셔너리 키 | `"W"`, `"b"` 같은 이름이다 |
| `val` | 딕셔너리 값 | 실제 파라미터 배열이다 |
| `SGD` | Stochastic Gradient Descent | 기본 Optimizer다 |
| `Momentum` | 관성 기반 Optimizer | velocity를 사용한다 |
| `v` | velocity 또는 2차 모멘트 | 문맥에 따라 다르다 |
| `RMSprop` | RMS 기반 Optimizer | gradient 제곱 평균을 사용한다 |
| `h` | gradient 제곱 이동 평균 | RMSprop에서 사용한다 |
| `decay_rate` | 감쇠율 | 이전 값을 얼마나 유지할지 정한다 |
| `epsilon` | 작은 값 | 0으로 나누는 것을 막는다 |
| `Adam` | Momentum + RMSprop | m과 v를 함께 사용한다 |
| `m` | 1차 모멘트 | gradient 이동 평균이다 |
| `beta1` | m의 감쇠율 | 보통 0.9다 |
| `beta2` | v의 감쇠율 | 보통 0.999다 |
| `m_hat` | 편향 보정된 m | Adam update에 사용한다 |
| `v_hat` | 편향 보정된 v | Adam update에 사용한다 |
| `AdamW` | Adam + weight decay | 가중치 감쇠를 분리 적용한다 |
| `weight_decay` | 가중치 감쇠 | 파라미터 크기를 줄인다 |

## 22. 핵심 코드 패턴 정리

### 22-1. 기본 경사하강법 패턴

```python
dW, db = gradient(X, y, W, b)
W = W - learning_rate * dW
b = b - learning_rate * db
```

기울기를 구하고 반대 방향으로 이동한다.

### 22-2. Optimizer 클래스 패턴

```python
class Optimizer:
    def __init__(self, learning_rate):
        self.lr = learning_rate

    def update(self, params, grads):
        for key in params.keys():
            params[key] -= self.lr * grads[key]
```

`params`와 `grads`를 받아 파라미터를 수정한다.

### 22-3. Momentum 패턴

```python
v = momentum * v - lr * grad
param = param + v
```

이전 이동 방향을 기억한다.

### 22-4. RMSprop 패턴

```python
h = decay_rate * h + (1 - decay_rate) * grad**2
param = param - lr * grad / (sqrt(h) + epsilon)
```

gradient 크기에 따라 보폭을 조절한다.

### 22-5. Adam 패턴

```python
m = beta1 * m + (1 - beta1) * grad
v = beta2 * v + (1 - beta2) * grad**2
m_hat = m / (1 - beta1**t)
v_hat = v / (1 - beta2**t)
param = param - lr * m_hat / (sqrt(v_hat) + epsilon)
```

Momentum과 RMSprop을 함께 사용한다.

### 22-6. AdamW 패턴

```python
param = param - lr * weight_decay * param
param = param - lr * m_hat / (sqrt(v_hat) + epsilon)
```

weight decay를 gradient update와 분리한다.

## 23. 시험용 요약

```text
Optimizer = gradient를 이용해 parameter를 어떻게 업데이트할지 정하는 규칙
```

꼭 기억할 것:

- 경사하강법은 손실이 작아지는 방향으로 파라미터를 이동시키는 방법이다.
- 기본 식은 `param = param - lr × grad`다.
- `W`는 weight, `b`는 bias다.
- `cost_function`은 현재 파라미터의 손실을 계산한다.
- 이번 예제의 cost는 MSE다.
- `gradient` 함수는 `dW`, `db`를 계산한다.
- `dW`는 W에 대한 미분값이다.
- `db`는 b에 대한 미분값이다.
- `learning_rate`가 너무 작으면 학습이 느리다.
- `learning_rate`가 너무 크면 발산할 수 있다.
- `params`는 파라미터 딕셔너리다.
- `grads`는 기울기 딕셔너리다.
- SGD는 현재 gradient만 보고 이동한다.
- Momentum은 이전 이동 방향을 velocity로 기억한다.
- RMSprop은 gradient 제곱의 이동 평균으로 보폭을 조절한다.
- Adam은 Momentum과 RMSprop을 결합한 Optimizer다.
- Adam의 `m`은 gradient의 이동 평균이다.
- Adam의 `v`는 gradient 제곱의 이동 평균이다.
- `m_hat`, `v_hat`은 편향 보정된 값이다.
- AdamW는 Adam에 weight decay를 분리 적용한 방식이다.
- `weight_decay`는 파라미터가 지나치게 커지는 것을 막는다.
- `epsilon`은 0으로 나누는 것을 막는 작은 값이다.